# CodeLlama + LoRA + RAG on Chinook

Schema-aware Text-to-SQL experiment with reusable project classes.

## 1. Project setup

In [ ]:
from pathlib import Path

PROJECT_ROOT = next((p for root in [Path.cwd(), Path('/content/project'), Path('/content')] for p in [root, *root.glob('**/*')] if p.is_dir() and (p / 'src').is_dir() and (p / 'requirements.txt').is_file()), None)
assert PROJECT_ROOT is not None, 'Extract the project ZIP under /content first.'
%cd $PROJECT_ROOT
print('Project:', PROJECT_ROOT)

In [ ]:
!pip -q install -r requirements.txt
print('Restart the session once only if Colab reports that imported packages were replaced.')

## 2. Load data and schema

In [ ]:
import json
from pathlib import Path
from transformers import set_seed
from src import ChinookData, ChinookRetriever, CodeLlamaRunner, ExperimentRunner, SQLEvaluator, evaluate_retriever

set_seed(42)
ROOT = Path.cwd()
DATA = ChinookData(ROOT / 'datasets/chinook')
train_examples, test_examples = DATA.load_train(), DATA.load_test()
schema, schema_text = DATA.schema_from_training(), DATA.full_schema_text()
print(len(train_examples), 'training |', len(test_examples), 'test |', len(schema), 'tables')

## 3. Load CodeLlama with the LoRA adapter

In [ ]:
ADAPTER = ROOT / 'models/codellama_chinook_lora'
runner = CodeLlamaRunner.load_adapter(ADAPTER)
evaluator = SQLEvaluator(DATA.db_path)
experiment = ExperimentRunner(runner, evaluator)
print('Model and adapter loaded')

## 4. Baseline evaluation

In [ ]:
baseline_rows, baseline_metrics = experiment.run(
    test_examples, schema_text=schema_text, corrected_sql_extraction=True)
baseline_metrics

## 5. RAG evaluation

In [ ]:
corpus = ChinookRetriever.build_corpus(schema, train_examples, example_limit=50)
retriever = ChinookRetriever(corpus)
rag_rows, rag_metrics = experiment.run(
    test_examples, retriever=retriever, top_k=4, corrected_sql_extraction=True)
rag_metrics

## 5A. Retriever evaluation (does not change RAG generation)

These component-level metrics use the gold SQL only to derive the tables required by each question. The existing retriever is called exactly as before. `Table Recall@K` measures required-table coverage, `Table Precision@K` measures how focused the retrieved table set is, and `MRR` measures how early the first chunk containing a required table appears.


In [ ]:
retrieval_rows, retrieval_metrics = evaluate_retriever(
    test_examples, retriever, top_k=4
)
retrieval_metrics

retrieval_output = ROOT / 'results/codellama_chinook_retrieval_metrics.json'
retrieval_output.write_text(json.dumps({
    'metrics': retrieval_metrics,
    'results': retrieval_rows,
}, indent=2, default=str))
print('Saved:', retrieval_output)


## 6. Save results

In [ ]:
output = ROOT / 'results/codellama_chinook_rag.json'
output.write_text(json.dumps({'results': rag_rows, 'metrics': rag_metrics}, indent=2))
print('Saved:', output)